# C2 refresh (leak-free retrieval) · Qwen/Qwen2.5-Coder-1.5B

Regenerates **only C2 always-retrieve** for all four datasets after the
retrieval-corpus fix (the completion's own file is now excluded from the
BM25 corpus, so the gold can no longer appear in the retrieved context).

GPU: Fits T4/L4.

Setup: `Runtime → A100 high-RAM`; Colab Secret `LUCA_GITHUB_PAT`.

## 1. Config

In [ ]:
# ---- experiment knobs (C2 refresh: leak-free retrieval corpus) ----
SMOKE = False
SMOKE_LIMIT = 32

MODEL = 'Qwen/Qwen2.5-Coder-1.5B'
MODEL_FAMILY = 'qwen'
RESULTS_TAG = 'qwen25_1.5b'

DATASETS = ['crosscodeeval_py', 'repoeval_function',
            'crosscodelongeval_function', 'crosscodelongeval_chunk']
MAX_TOKENS = {'crosscodeeval_py': 50, 'repoeval_function': 280,
              'crosscodelongeval_function': 400, 'crosscodelongeval_chunk': 80}
TOP_K = 10
BATCH_SIZE = 256
GEN_CONFIGS = ['C2_always_retrieve']    # ONLY C2: zero-shot side (C1/C3) is unchanged

# ---- git (REST API; no clone/push) ----
REPO = 'Luca-Ionescu/rag-static-analysis-experiment'
SRC_REF = 'main'
GH_RESULTS_BRANCH = 'colab-results'
WORK_DIR = '/content/rag-static-analysis-experiment'
print('C2-REFRESH | model', MODEL, '| datasets', len(DATASETS))


## 2. GPU sanity

In [ ]:
import subprocess
try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as e:
    print('No GPU — Runtime -> Change runtime type -> GPU.', e)

## 3. GitHub token + REST helpers

In [ ]:
import os, sys, base64, json as _gjson, urllib.request, urllib.error

def _get_pat():
    try:
        from google.colab import userdata
        pat = userdata.get('LUCA_GITHUB_PAT')
        if pat: return pat
    except Exception as e:
        print('  secret read:', e)
    pat = os.environ.get('LUCA_GITHUB_PAT')
    if pat: return pat
    import getpass
    return getpass.getpass('LUCA_GITHUB_PAT (Contents: read/write): ').strip()

_GH_PAT = _get_pat(); assert _GH_PAT, 'no PAT provided'

def _gh_req(method, url, data=None, raw=False):
    req = urllib.request.Request(url, method=method)
    req.add_header('Authorization', f'Bearer {_GH_PAT}')
    req.add_header('Accept', 'application/vnd.github.v3.raw' if raw else 'application/vnd.github+json')
    req.add_header('User-Agent', 'colab-runner')
    req.add_header('X-GitHub-Api-Version', '2022-11-28')
    body = None
    if data is not None:
        body = _gjson.dumps(data).encode(); req.add_header('Content-Type', 'application/json')
    with urllib.request.urlopen(req, body, timeout=180) as resp:
        out = resp.read()
        return out if raw else _gjson.loads(out.decode())

print('authenticated as:', _gh_req('GET', 'https://api.github.com/user').get('login'))


## 4. Pull repo (tarball REST API)

In [ ]:
import os, io, tarfile, shutil
if os.path.isdir(WORK_DIR): shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)
blob = _gh_req('GET', f'https://api.github.com/repos/{REPO}/tarball/{SRC_REF}', raw=True)
with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as tar:
    root = tar.getmembers()[0].name.split('/')[0]
    tar.extractall('/content/_repo_extract')
for name in os.listdir(f'/content/_repo_extract/{root}'):
    shutil.move(f'/content/_repo_extract/{root}/{name}', f'{WORK_DIR}/{name}')
shutil.rmtree('/content/_repo_extract'); os.chdir(WORK_DIR)
print('repo at', os.getcwd())


## 5. Install dependencies

In [ ]:
import subprocess, sys
pkgs = ['vllm==0.10.2', 'transformers>=4.55.2,<5.0',
        'tree-sitter==0.23.2', 'tree-sitter-python==0.23.6',
        'rank-bm25', 'python-Levenshtein', 'lightgbm', 'jsonlines', 'click',
        'pyflakes', 'tqdm', 'scipy', 'scikit-learn', 'datasets', 'huggingface-hub']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)
print('deps installed')


## 6. HF token (public models; secret optional)

In [ ]:
# HF token: required for the-stack-dedup (gated) used by 0.5B calibration; the
# code models themselves are public. Reads Colab secret HF_TOKEN if present.
import os
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
    if tok: os.environ['HF_TOKEN'] = tok
except Exception:
    pass
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))


## 7. Provision CCE + RepoEval

In [ ]:
import os, subprocess, zipfile, glob
os.chdir(WORK_DIR)
# CrossCodeEval (committed gzip asset)
os.makedirs('data/crosscodeeval/crosscodeeval_data/python', exist_ok=True)
cce = 'data/crosscodeeval/crosscodeeval_data/python/line_completion_rg1_bm25.jsonl'
if not os.path.exists(cce):
    subprocess.run(f'gunzip -c scripts/runpod/assets/cce_python_rg1_bm25.jsonl.gz > {cce}', shell=True, check=True)
print('CCE instances:', sum(1 for _ in open(cce)))
# RepoEval-function (microsoft/CodeT RepoCoder)
os.makedirs('data/repoeval/datasets', exist_ok=True)
os.makedirs('data/repoeval/repositories', exist_ok=True)
if not os.path.exists('/content/CodeT'):
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse',
                    'https://github.com/microsoft/CodeT.git','/content/CodeT'], check=True)
    subprocess.run(['git','-C','/content/CodeT','sparse-checkout','set','RepoCoder'], check=True)
RC = '/content/CodeT/RepoCoder'
with zipfile.ZipFile(f'{RC}/datasets/datasets.zip') as z: z.extractall('data/repoeval/datasets')
with zipfile.ZipFile(f'{RC}/repositories/function_level.zip') as z: z.extractall('data/repoeval/repositories')
print('RepoEval function task:', glob.glob('data/repoeval/datasets/function_level_completion_2k*.jsonl'))


## 7b. Provision CrossCodeLongEval

In [ ]:
import os, subprocess, tarfile, glob
os.chdir(WORK_DIR)
os.makedirs('data/crosscodelongeval', exist_ok=True)
if not os.path.exists('/content/Repoformer'):
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse',
                    'https://github.com/amazon-science/Repoformer.git','/content/Repoformer'], check=True)
    subprocess.run(['git','-C','/content/Repoformer','sparse-checkout','set','crosscodelongeval'], check=True)
RF = '/content/Repoformer/crosscodelongeval'
for tb in ('cceval_function_eval_data.tar.gz', 'cceval_chunk_eval_data.tar.gz'):
    with tarfile.open(f'{RF}/{tb}') as t: t.extractall('data/crosscodelongeval')
fn = glob.glob('data/crosscodelongeval/cceval_function_eval_data/*sparse_rg1*.jsonl')
ch = glob.glob('data/crosscodelongeval/cceval_chunk_eval_data/*sparse_rg1*.jsonl')
print('CCLE function:', fn, '->', sum(1 for _ in open(fn[0])) if fn else 0, 'instances')
print('CCLE chunk   :', ch, '->', sum(1 for _ in open(ch[0])) if ch else 0, 'instances')


## 8. Results branch + uploader (REST)

In [ ]:
def _gh_ensure_branch():
    refbase = f'https://api.github.com/repos/{REPO}/git/refs/heads/'
    try:
        _gh_req('GET', refbase + GH_RESULTS_BRANCH); return
    except urllib.error.HTTPError as e:
        if e.code != 404: raise
    info = _gh_req('GET', f'https://api.github.com/repos/{REPO}')
    head = _gh_req('GET', refbase + info['default_branch'])
    _gh_req('POST', f'https://api.github.com/repos/{REPO}/git/refs',
            {'ref': f'refs/heads/{GH_RESULTS_BRANCH}', 'sha': head['object']['sha']})

def _gh_get_sha(dest):
    try:
        return _gh_req('GET', f'https://api.github.com/repos/{REPO}/contents/{dest}?ref={GH_RESULTS_BRANCH}').get('sha')
    except Exception:
        return None

def gh_upload(path, prefix):
    from pathlib import Path as _P
    p = _P(path)
    if not p.exists():
        print('  skip (missing):', path); return
    dest = f'{prefix}/{p.name}'
    payload = {'message': f'colab: {dest}', 'content': base64.b64encode(p.read_bytes()).decode(),
               'branch': GH_RESULTS_BRANCH}
    sha = _gh_get_sha(dest)
    if sha: payload['sha'] = sha
    _gh_req('PUT', f'https://api.github.com/repos/{REPO}/contents/{dest}', payload)
    print('  pushed:', dest)

_gh_ensure_branch(); print('results branch ready:', GH_RESULTS_BRANCH)


## 9. Generate C2 per dataset (push each)

In [ ]:
import os, subprocess, sys
os.chdir(WORK_DIR)
# Sanity: the leak fix must be present in the pulled source.
src = open('src/adaptive_retrieval/retriever.py').read()
assert 'exclude_file' in src, 'retriever fix missing on SRC_REF!'
src2 = open('src/adaptive_retrieval/eval/runner.py').read()
assert 'exclude_file=inst.target_file' in src2, 'runner fix missing on SRC_REF!'
print('leak-fix present in source: OK')

def run_config(cfg, ds, subdir):
    out = f'{subdir}/{cfg}.jsonl'
    cmd = [sys.executable, 'scripts/04_run_experiment.py',
           '--config', cfg, '--dataset', ds, '--backend', 'vllm',
           '--model', MODEL, '--model-family', MODEL_FAMILY,
           '--max-tokens', str(MAX_TOKENS[ds]),
           '--top-k', str(TOP_K),
           '--batch-size', str(BATCH_SIZE),
           '--output', out, '--cache-dir', 'data/generation_cache']
    if SMOKE:
        cmd += ['--limit', str(SMOKE_LIMIT)]
    print('>>>', ' '.join(cmd)); subprocess.run(cmd, check=True)
    gh_upload(out, subdir)

for ds in DATASETS:
    subdir = f'results/{RESULTS_TAG}_{ds}'
    os.makedirs(subdir, exist_ok=True)
    for cfg in GEN_CONFIGS:
        run_config(cfg, ds, subdir)
print('C2 refresh done (all datasets pushed)')
